# laser-ai training notebook

**Fresh training:** upload `bundle.zip` and run all cells.

**Resume sequencer-only (skip VAE):** also upload an existing `model.pt` (or `vae_only.pt`). The notebook will detect it and skip VAE training, training the sequencer for `seq_epochs=200` on standardized latents.

In [ ]:
# 1. Install laser-ai from GitHub (force-reinstall to bypass pip cache)
!pip uninstall -y laser-ai
!pip install -q --upgrade pip
!pip install -q --no-cache-dir git+https://github.com/tobytorgerson-art/laser-ai.git

In [ ]:
# 2. Detect uploads
import pathlib
zips = list(pathlib.Path('.').glob('*.zip'))
assert zips, 'Upload bundle.zip to the file panel first'
BUNDLE = str(zips[0])

# Resume if any of these names are present in the working dir
RESUME = None
for name in ('model.pt', 'vae_only.pt'):
    if pathlib.Path(name).exists():
        RESUME = name
        break

print('bundle:', BUNDLE)
print('resume:', RESUME if RESUME else '(none — full fresh run)')

In [ ]:
# 3. Sanity check: confirm installed code has the standardization fix
import importlib, inspect, laser_ai.training.train_sequencer as ts
importlib.reload(ts)
src = inspect.getsource(ts.train_sequencer)
assert 'latent_mean' in src and 'lats_norm' in src, 'Stale install — restart runtime and rerun cell 1'
print('OK: standardization code present')

In [ ]:
# 4. Run training
from laser_ai.colab_train import run
OUT = 'model_new.pt' if RESUME else 'model.pt'
if RESUME:
    # Sequencer-only: longer run since VAE is skipped (~minutes vs hours)
    run(BUNDLE, OUT, resume_from_vae=RESUME, seq_epochs=200)
else:
    run(BUNDLE, OUT)
print('\nTraining complete. Download', OUT, 'from the file panel.')